# E2 current ligand-source receiving

Prospective source admission for uibcdf/dockingmt#81. Execute only with the
registered installed worker/resource supervisor and DOCKINGMT_E2_ROOT,
DOCKINGMT_E2_OUTPUT, DOCKINGMT_E2_CANDIDATE_MANIFEST and its _SHA256 variable.

Receive both pinned SDFs using public MolSysMT. The initial native may be partial;
native → RDKit → native creates a new interpreted authority. Preserve coverage,
raw SD properties and the actual provider wedge/3D interpretation. No additional
coordinate-based stereo inference or initial-to-new full-graph equivalence.

Direct caps: SDF 2, native-to-RDKit 2, back-to-native 2, readiness 4, explicit CIP 2,
new-authority identity-candidate self-witnesses 2. Internal provider conversions/
CIP and inspection calls are outside those caps and observed separately. SDF
stereo reads themselves invoke CIP. Graph completeness concerns one supplied
identity candidate, without symmetry enumeration. All work uses pm/fs/coulomb.

Stop the entire two-candidate selection on failure, preserving reports, partial
outcomes and not-attempted members. No substitutions or retries. No preparation,
new H, charge/type assignment, Vina import, energy, search, comparison or H5 IO.
Inspection JSON is evidence, never a molecular/MM restoration codec. Valence,
protonation, experimental H, activity, affinity and whole E2 remain unassessed.


In [ ]:
import hashlib
import inspect
import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path, PurePosixPath

root = Path(os.environ['DOCKINGMT_E2_ROOT']).resolve()
output = Path(os.environ['DOCKINGMT_E2_OUTPUT']).resolve()
assert not output.exists() and not root.is_relative_to(output)
output.mkdir(parents=True, exist_ok=False)
manifest_path = Path(os.environ['DOCKINGMT_E2_CANDIDATE_MANIFEST']).resolve()
manifest_sha = os.environ['DOCKINGMT_E2_CANDIDATE_MANIFEST_SHA256']
expected_manifest_sha = (
    '898b26fa60561bb99f823d2dd0c27216cc5771ac7dceacdb103760fed2e22284'
)
expected_candidates = [
    dict(
        ligand_id='p59',
        state_id='e2-5x72-p59',
        path='tests/data/vina_torsions/5x72_ligand_p59H.sdf',
        sha256='ef26c05a198a16972efcf8baae644f6b38fd953561f130ee230626088766b08a',
        expected_cip='R',
        source_stereo_atom_index=7,
    ),
    dict(
        ligand_id='p69',
        state_id='e2-5x72-p69',
        path='tests/data/vina_torsions/5x72_ligand_p69H.sdf',
        sha256='637a992134b2038a0ea3cff0c1f8355ea232720c9eae5347100b7186170fcdf1',
        expected_cip='S',
        source_stereo_atom_index=7,
    ),
]
budgets = dict(
    sdf_native=2,
    native_to_rdkit=2,
    rdkit_to_native=2,
    readiness=4,
    explicit_cip=2,
    self_witness=2,
)
counts = dict.fromkeys(budgets, 0)
observed = dict(
    public_convert_alias=0,
    public_get_alias=0,
    snapshot=0,
    public_cip_alias=0,
    public_readiness_alias=0,
    public_correspondence_alias=0,
)
forbidden_attempts = []
ledger = [
    dict(
        position=i + 1,
        ligand_id=c['ligand_id'],
        candidate_state_id=c['state_id'],
        status='not_attempted',
        stages={},
    )
    for i, c in enumerate(expected_candidates)
]
current = None
stage = 'custody'


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def save_json(path, value):
    payload = json.dumps(value, indent=2, allow_nan=False) + '\n'
    with path.open('x') as stream:
        stream.write(payload)


def persist_ledger():
    (output / 'ledger.json').write_text(
        json.dumps(ledger, indent=2, allow_nan=False) + '\n'
    )


def event(operation, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    operation=operation,
                    stage=stage,
                    candidate=None if current is None else current['ligand_id'],
                    at_utc=datetime.now(timezone.utc).isoformat(),
                    direct_counts=counts.copy(),
                    **fields,
                ),
                allow_nan=False,
            )
            + '\n'
        )


persist_ledger()
try:
    raw_manifest = manifest_path.read_bytes()
    (output / 'candidate_manifest.json').write_bytes(raw_manifest)
    assert (
        manifest_sha
        == expected_manifest_sha
        == hashlib.sha256(raw_manifest).hexdigest()
    )
    candidate_manifest = json.loads(raw_manifest)
    assert candidate_manifest['schema'] == 'dockingmt.e2_ligand_source_candidates@1'
    assert candidate_manifest['owner'] == 'uibcdf/dockingmt#81'
    assert len(candidate_manifest['candidates']) == 2
    assert candidate_manifest['source_license']['spdx'] == 'Apache-2.0'
    assert (
        candidate_manifest['source_license']['commit']
        == '3c65c0b3e6c2c1d183f6a175ecb65e3c5ba91645'
    )
    for actual, expected in zip(candidate_manifest['candidates'], expected_candidates):
        assert {key: actual[key] for key in expected} == expected
    stored_inputs = {str(manifest_path): manifest_sha}
    input_copies = {}
    inputs = output / 'inputs'
    inputs.mkdir()
    assert len(candidate_manifest['files']) == 7
    seen = set()
    for row in candidate_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        assert str(relative) == row['path'] and row['path'] not in seen
        seen.add(row['path'])
        path = root / relative
        assert path.resolve().is_relative_to(root)
        data = path.read_bytes()
        copy_path = inputs / relative.name
        copy_path.write_bytes(data)
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        input_copies[str(copy_path)] = row['sha256']
    for row in expected_candidates:
        assert stored_inputs[str(root / row['path'])] == row['sha256']
    assert os.sched_getaffinity(0) == {0} and os.getpriority(os.PRIO_PROCESS, 0) >= 10
    meminfo = dict(
        line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()
    )
    assert int(meminfo['MemAvailable'].split()[0]) * 1024 >= 8 * 1024**3
    assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_source_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            retry_permitted=False,
        ),
    )
    raise

In [ ]:
class RejectVinaImports:
    def find_spec(self, fullname, path=None, target=None):
        if fullname.split('.')[0] == 'vina':
            forbidden_attempts.append('import:' + fullname)
            raise AssertionError('Vina import is forbidden in source admission')
        return None


import_guard = RejectVinaImports()
sys.meta_path.insert(0, import_guard)
try:
    import importlib.metadata as metadata

    import molsysmt as msm
    import numpy as np
    import pyunitwizard as puw

    import dockingmt as dmt
    from dmt_test_support.qualify_chemical_templates import detached_record, snapshot
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_source_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage='imports',
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            forbidden_attempts=forbidden_attempts,
            retry_permitted=False,
        ),
    )
    raise

patches = []
records = []
live_sources = []
identities = {}
installed_check = globals()['registered_installed_check']
caller_policy = detached_record(puw.configure.report())
integrity = dict(before=False, after=False)


def save(path, value):
    save_json(path, detached_record(value))


def patch(owner, name, replacement):
    original = getattr(owner, name)
    patches.append((owner, name, original))
    setattr(owner, name, replacement)


def watch(function, counter):
    @wraps(function)
    def observed_call(*args, **kwargs):
        observed[counter] += 1
        return function(*args, **kwargs)

    return observed_call


def reject(name):
    def forbidden(*args, **kwargs):
        forbidden_attempts.append(name)
        event('forbidden', operation_name=name)
        raise AssertionError('Forbidden source-admission operation: ' + name)

    return forbidden


def install_monitoring():
    for owner, name, counter in (
        (msm, 'convert', 'public_convert_alias'),
        (msm, 'get', 'public_get_alias'),
        (msm.physchem, 'get_cip_stereochemistry', 'public_cip_alias'),
        (msm.physchem, 'get_chemical_readiness', 'public_readiness_alias'),
        (
            msm.chemical_states,
            'get_atom_correspondences',
            'public_correspondence_alias',
        ),
    ):
        patch(owner, name, watch(getattr(owner, name), counter))
    for owner, name in (
        (dmt, 'dock'),
        (dmt, 'score'),
        (dmt, 'dock_many'),
        (dmt, 'prepare_ligand'),
        (dmt, 'prepare_receptor'),
        (dmt, 'compare_scores'),
        (dmt.VinaBackend, 'dock'),
        (dmt.VinaBackend, 'score'),
        (msm.build, 'add_missing_hydrogens'),
        (msm.build, 'assign_partial_charges'),
        (msm.build, 'assign_autodock_atom_types'),
    ):
        patch(owner, name, reject(name))


def direct(label, budget, function, *args, **kwargs):
    global stage
    stage = label
    assert counts[budget] < budgets[budget], budget
    counts[budget] += 1
    current['stages'][label] = 'started'
    persist_ledger()
    event('started', direct_operation=budget)
    value = function(*args, **kwargs)
    current['stages'][label] = 'returned'
    persist_ledger()
    event('returned', direct_operation=budget)
    return value


def inspect_native(system):
    observed['snapshot'] += 1
    chemistry = snapshot(system)
    return detached_record(
        dict(
            form=msm.get_form(system),
            chemistry_structures=chemistry,
            n_atoms=int(msm.get(system, n_atoms=True)),
            n_bonds=int(msm.get(system, n_bonds=True)),
            ids=[str(x) for x in msm.get(system, element='atom', atom_id=True)],
            elements=[
                str(x)
                for x in msm.get(
                    system,
                    element='atom',
                    atom_type=True,
                    structure_indices=0,
                    chemical_state='structure',
                )
            ],
            coordinates_angstrom=puw.get_value(
                msm.get(system, coordinates=True), to_unit='angstrom'
            ),
            coordinate_unit='angstrom',
            has_partial_charge=bool(msm.has_attribute(system, 'partial_charge')),
            has_atom_ff_type=bool(msm.has_attribute(system, 'atom_ff_type')),
        )
    )


def source_identity():
    for name, module in (
        ('dockingmt', dmt),
        ('molsysmt', msm),
        ('numpy', np),
        ('pyunitwizard', puw),
    ):
        path = Path(module.__file__).resolve()
        identities[name] = dict(
            version=metadata.version(name), path=str(path), sha256=file_sha(path)
        )
    for name, function in (
        ('convert', msm.convert),
        ('get', msm.get),
        ('chemical_readiness', msm.physchem.get_chemical_readiness),
        ('cip', msm.physchem.get_cip_stereochemistry),
        ('atom_correspondences', msm.chemical_states.get_atom_correspondences),
        ('snapshot', snapshot),
        ('detached_record', detached_record),
    ):
        function = inspect.unwrap(function)
        path = Path(function.__code__.co_filename).resolve()
        identities[name] = dict(
            module=function.__module__, path=str(path), sha256=file_sha(path)
        )
    helper_sha = file_sha(root / 'dmt_test_support/qualify_chemical_templates.py')
    assert (
        identities['snapshot']['sha256']
        == identities['detached_record']['sha256']
        == helper_sha
    )
    save(output / 'initial_module_identities.json', identities)


def receive_candidate(candidate):
    global stage
    directory = output / candidate['ligand_id']
    directory.mkdir()
    source_file = inputs / Path(candidate['path']).name
    native, report = direct(
        'sdf_native',
        'sdf_native',
        msm.convert,
        str(source_file),
        to_form='molsysmt.MolSys',
        stereo_engine='rdkit',
        discard_properties=True,
        return_report=True,
    )
    save(directory / 'source_conversion_report.json', asdict(report))
    native_before = inspect_native(native)
    save(directory / 'source_native_inspection.json', native_before)
    live_sources.append((native, native_before))
    before = direct(
        'source_readiness',
        'readiness',
        msm.physchem.get_chemical_readiness,
        native,
        selection='all',
        structure_indices=0,
        chemical_state='structure',
    )
    before = detached_record(before)
    save(directory / 'source_readiness.json', before)
    rdkit_model, report = direct(
        'native_to_rdkit',
        'native_to_rdkit',
        msm.convert,
        native,
        to_form='rdkit.Mol',
        return_report=True,
    )
    save(directory / 'native_to_rdkit_report.json', asdict(report))
    intermediate = detached_record(
        dict(
            form=msm.get_form(rdkit_model),
            n_atoms=int(msm.get(rdkit_model, n_atoms=True)),
            n_bonds=int(msm.get(rdkit_model, n_bonds=True)),
            coordinates_angstrom=puw.get_value(
                msm.get(rdkit_model, coordinates=True), to_unit='angstrom'
            ),
            coordinate_unit='angstrom',
            scope='Current RDKit interpretation; no direct toolkit edit or serialization',
        )
    )
    save(directory / 'rdkit_intermediate_inspection.json', intermediate)
    authority, report = direct(
        'rdkit_to_native',
        'rdkit_to_native',
        msm.convert,
        rdkit_model,
        to_form='molsysmt.MolSys',
        return_report=True,
    )
    save(directory / 'rdkit_to_native_report.json', asdict(report))
    authority_before = inspect_native(authority)
    save(directory / 'authority_native_inspection.json', authority_before)
    live_sources.append((authority, authority_before))
    after = direct(
        'authority_readiness',
        'readiness',
        msm.physchem.get_chemical_readiness,
        authority,
        selection='all',
        structure_indices=0,
        chemical_state='structure',
    )
    after = detached_record(after)
    save(directory / 'authority_readiness.json', after)
    cip = direct(
        'authority_cip',
        'explicit_cip',
        msm.physchem.get_cip_stereochemistry,
        authority,
        selection='all',
        structure_indices=0,
        chemical_state='structure',
        engine='rdkit',
        from_coordinates=False,
    )
    cip = detached_record(cip)
    save(directory / 'authority_cip.json', cip)
    # Receive the fixture axis before supplying its graph candidate. This is not
    # complete chemical equivalence across a declared interpretation.
    stage = 'source_axis_and_geometry_checks'
    for inspected in (native_before, authority_before):
        assert inspected['n_atoms'] == 39 and inspected['n_bonds'] == 42
        assert len(inspected['ids']) == len(set(inspected['ids'])) == 39
        assert inspected['ids'] == [str(i) for i in range(1, 40)]
        assert (
            len(inspected['elements']) == 39 and inspected['elements'].count('H') == 15
        )
        assert not inspected['has_partial_charge'] and not inspected['has_atom_ff_type']
    assert authority_before['ids'] == native_before['ids']
    assert authority_before['elements'] == native_before['elements']
    assert intermediate['form'] == 'rdkit.Mol'
    assert intermediate['n_atoms'] == 39 and intermediate['n_bonds'] == 42
    for coordinates in (
        intermediate['coordinates_angstrom'],
        authority_before['coordinates_angstrom'],
    ):
        np.testing.assert_allclose(
            coordinates, native_before['coordinates_angstrom'], rtol=0, atol=1e-12
        )
    np.testing.assert_array_equal(
        before['bonded_atom_pairs'], after['bonded_atom_pairs']
    )
    save(
        directory / 'axis_geometry_receiving.json',
        dict(
            source_atom_ids=native_before['ids'],
            authority_atom_ids=authority_before['ids'],
            received_ordered_axis=[[i, i] for i in range(39)],
            evidence='Public native IDs/elements and unchanged ordered connectivity/coordinates',
            coordinate_unit='angstrom',
            coordinate_tolerance=1e-12,
            source_to_authority_full_graph_equivalence_claimed=False,
        ),
    )
    stage = 'authority_coverage_and_stereo_checks'
    assert before['schema'] == after['schema'] == 'molsysmt.chemical_readiness@1'
    assert after['chemical_state_status'] == 'resolved' and after['n_atoms'] == 39
    assert after['connectivity']['declared_completeness'] == 'complete'
    assert after['connectivity']['invalid_bond_indices'] == []
    assert after['connectivity']['crossing_bond_indices'] == []
    assert len(after['explicit_hydrogen_atom_indices']) == 15
    required = (
        'atom_id',
        'atom_type',
        'formal_charge',
        'atom_is_aromatic',
        'n_unpaired_electrons',
        'n_implicit_hydrogens',
        'n_explicit_hydrogens',
        'allows_implicit_hydrogens',
        'atom_stereochemistry',
        'covalent_multiplicity',
        'coordinates',
    )
    for name in required:
        assert after['fields'][name]['status'] == 'present', name
    for name in ('n_implicit_hydrogens', 'n_explicit_hydrogens'):
        assert after['fields'][name]['values'] == [0] * 39, name
    assert after['fields']['coordinates']['values'] == [True] * 39
    assert {'valence', 'protonation', 'conformer_quality', 'property_origin'} <= set(
        after['unassessed_checks']
    )
    assert cip['method'] == 'hanson_2018' and cip['engine'] == 'rdkit'
    assert cip['evidence'] == 'declared_stereochemistry'
    assert cip['atom_indices'] == list(range(39))
    assert (
        cip['atom_stereochemistry'][candidate['source_stereo_atom_index']]
        == candidate['expected_cip']
    )
    raw_witness = direct(
        'authority_self_witness',
        'self_witness',
        msm.chemical_states.get_atom_correspondences,
        authority,
        reference_molecular_system=authority,
        selection='all',
        reference_selection='all',
        structure_indices=0,
        reference_structure_indices=0,
        chemical_state='structure',
        reference_chemical_state='structure',
        candidate_atom_correspondences=[[[i, i] for i in range(39)]],
        max_matches=1,
    )
    witness = detached_record(raw_witness)
    save(directory / 'authority_self_witness.json', witness)
    assert witness['schema'] == 'molsysmt.atom_correspondences@1'
    assert witness['method'] == 'declared_graph_isomorphism'
    assert witness['search_scope'] == 'supplied_atom_correspondences'
    assert witness['search_complete'] is True and witness['n_candidate_matches'] == 1
    assert witness['evidence']['kind'] == 'declared_complete_covalent_graph'
    assert witness['atom_correspondences'] == [[[i, i] for i in range(39)]]
    assert witness['source_graph_digest'] == witness['reference_graph_digest']
    pristine = detached_record(witness)
    # Mutate the actual public return after preserving its finite JSON record.
    raw_witness['software']['molsysmt'] = 'detachment-control'
    raw_witness['atom_correspondences'][0, 0, 0] = -1
    assert inspect_native(authority) == authority_before
    assert inspect_native(native) == native_before
    witness = pristine
    coverage = {
        name: dict(before=field['status'], after=after['fields'][name]['status'])
        for name, field in before['fields'].items()
    }
    record = dict(
        ligand_id=candidate['ligand_id'],
        candidate_state_id=candidate['state_id'],
        current_authority_admission='ADMITTED',
        source_sha256=candidate['sha256'],
        source_stereo_atom_index=candidate['source_stereo_atom_index'],
        actual_cip=cip['atom_stereochemistry'][candidate['source_stereo_atom_index']],
        source_property_policy='discard_properties=True for native storage; complete raw SDF retained',
        interpretation='Public native SDF stereo_engine=rdkit, then explicit native/RDKit/native',
        full_graph_equivalence_to_initial_native_claimed=False,
        source_hydrogen_atoms=15,
        generated_hydrogens=0,
        coverage=coverage,
        unassessed_checks=after['unassessed_checks'],
        authority_graph_digest=witness['source_graph_digest'],
        graph_scope='One supplied full-axis identity candidate; no symmetry enumeration',
        mechanics_restored=False,
        preparation_received=False,
        whole_e2_received=False,
    )
    save(directory / 'admission.json', record)
    return record

In [ ]:
failure = None
outcome = 'FAIL'
try:
    installed_check('before-e2-source-admission')
    integrity['before'] = True
    source_identity()
    with puw.context(
        standard_units=[
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ]
    ):
        active_policy = detached_record(puw.configure.report())
        save(output / 'active_unit_policy.json', active_policy)
        install_monitoring()
        for position, candidate in enumerate(expected_candidates):
            current = ledger[position]
            current['status'] = 'attempted'
            persist_ledger()
            record = receive_candidate(candidate)
            records.append(record)
            current['status'] = 'admitted'
            current['admission_record'] = candidate['ligand_id'] + '/admission.json'
            persist_ledger()
            assert detached_record(puw.configure.report()) == active_policy
        stage = 'complete_selection_checks'
        assert len(records) == 2 and counts == budgets
        assert [row['actual_cip'] for row in records] == ['R', 'S']
        assert len({row['candidate_state_id'] for row in records}) == 2
        assert len({row['authority_graph_digest'] for row in records}) == 2
        for system, before in live_sources:
            assert inspect_native(system) == before
        for path, expected in {**stored_inputs, **input_copies}.items():
            assert file_sha(Path(path)) == expected
        assert not forbidden_attempts
        assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
        for name in (
            'rdkit',
            'rdkit.rdBase',
            'rdkit.Chem.rdchem',
            'rdkit.Chem.rdCIPLabeler',
            'molsysmt.form.file_sdf.to_molsysmt_MolSys',
            'molsysmt.form.molsysmt_MolSys.to_rdkit_Mol',
            'molsysmt.form.rdkit_Mol.to_molsysmt_MolSys',
        ):
            module = sys.modules[name]
            path = Path(module.__file__).resolve()
            identities[name] = dict(path=str(path), sha256=file_sha(path))
        save(output / 'received_module_identities.json', identities)
        save(
            output / 'admitted_sources.json',
            dict(
                schema='dockingmt.e2_declared_ligand_sources@1',
                candidates=records,
                distinct_declared_stereochemical_entities=2,
                experimental_complexes=1,
                source_authority_scope='Current interpreted authorities; no historical authentication or chemical-validity certificate',
                new_preparations=0,
                new_energies=0,
                new_searches=0,
                whole_e2_received=False,
            ),
        )
        outcome = 'PASS'
except BaseException as error:
    failure = error
    if current is not None:
        current['status'] = 'failed'
        current['failed_stage'] = stage
        persist_ledger()
    save(
        output / 'failure.json',
        dict(
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            code=getattr(error, 'code', None),
            direct_counts=counts,
            observed_public_alias_calls=observed,
            complete_denominator=2,
            retry_permitted=False,
        ),
    )
finally:
    for owner, name, original in reversed(patches):
        setattr(owner, name, original)
    if import_guard in sys.meta_path:
        sys.meta_path.remove(import_guard)
    try:
        assert detached_record(puw.configure.report()) == caller_policy
        installed_check('after-e2-source-admission')
        assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
        integrity['after'] = True
    except BaseException as error:
        save(
            output / 'integrity_failure.json',
            dict(error_type=type(error).__name__, reason=str(error)),
        )
        if failure is None:
            failure = error
        else:
            failure.add_note('Final integrity check failed: ' + str(error))
    if failure is not None:
        outcome = 'FAIL'
    persist_ledger()
    save(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_source_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome=outcome,
            candidate_manifest_sha256=manifest_sha,
            direct_operation_budgets=budgets,
            direct_operation_counts=counts,
            direct_count_scope='Explicit calls only; provider-internal conversions/CIP are not new caller requests',
            observed_public_alias_calls_and_snapshots=observed,
            observed_count_scope='Instrumented public aliases and qualification snapshots; not every internal provider dispatch',
            source_input_sha256=stored_inputs,
            raw_input_copy_sha256=input_copies,
            source_license=candidate_manifest['source_license'],
            module_identities=identities,
            installed_integrity_checks=integrity,
            caller_unit_policy=caller_policy,
            native_unit_policy='pm/fs/coulomb',
            caller_policy_restored=detached_record(puw.configure.report())
            == caller_policy,
            candidate_denominator=2,
            attempted=sum(row['status'] != 'not_attempted' for row in ledger),
            admitted=len(records),
            ledger=ledger,
            forbidden_attempts=forbidden_attempts,
            new_preparations=0,
            new_charge_assignments=0,
            new_type_assignments=0,
            new_hydrogens=0,
            new_energies=0,
            new_searches=0,
            vina_imported=any(name.split('.')[0] == 'vina' for name in sys.modules),
            h5_persistence_calls=0,
            retries=0,
            retry_permitted=False,
            inspection_json_is_restoration_input=False,
            initial_native_to_authority_full_graph_equivalence_claimed=False,
            actual_source_sdf_interpretation='RDKit-supported wedge/3D interpretation; no additional coordinate stereo inference',
            unassessed_science='Valence, protonation, experimental H, conformation, activity, receptor readiness, screening/affinity',
            whole_e2_received=False,
        ),
    )
if failure is not None:
    raise failure